# Phenotypes: plausible range, residualise, trim, standardise within sex

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run, repo_root
from aoucov.batch import run_script
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = cfg.params.get("threads", 4)

In [ ]:
from stage_helpers import hist_safe
STAGE = "phenotypes"
require_ready(cfg, STAGE)
rep = Report(cfg, STAGE)
items = target_items(cfg, STAGE)
OUT_DIR = os.path.dirname(items[0][0])

In [ ]:
if FORCE or stage_status(cfg, STAGE).missing:
    run_script(cfg, "residualize.R", dict(
        LIB=os.path.join(repo_root(), "scripts/local/residualize_lib.R"),
        TABLE_DIR=cfg.input("phenotype_tables"), PHENO_LIST=cfg.input("phenotype_list"),
        PHENOS=",".join(cfg.lists["phenotypes"]), TRANSFORMS=",".join(cfg.lists["transforms"]),
        COVSETS=",".join(cfg.lists["covsets"]), KEEP=single_target(cfg, "ancestry_round2"),
        PCS=single_target(cfg, "covariate_pcs"), N_PCS=cfg.param("cov_n_pcs"),
        OUTLIER_SD=cfg.param("outlier_sd"), OUT_DIR=OUT_DIR))
    for uri, item in items:
        write_meta(get_store(uri), uri, collect_meta(cfg, STAGE, uri, item=item, extra={"script": "residualize.R"}))

## Checks

In [ ]:
qc = pd.read_csv(f"{OUT_DIR}/qc_summary.tsv", sep="\t")
MIN_N = cfg.params.get("pheno_min_n", 1000)
for r in qc.itertuples():
    tag = f"{r.phenotype}/{r.transform}/{r.covset}"
    rep.check(f"{tag}: N retained >= {MIN_N}", r.n_retained >= MIN_N)
for ph, d in qc.groupby("phenotype"):
    r = d.iloc[0]
    rep.check(f"{ph}: not near-binary (>= 50 distinct values)", r.n_distinct >= 50)
    rep.check(f"{ph}: implausible values < 1%", r.n_implausible / r.n_in_set < 0.01,
              f"{r.n_implausible / r.n_in_set:.2%}")
for uri, item in items:
    y = pd.read_csv(uri, sep=" ")["Y"]
    ok = abs(y.mean()) < 0.05 and abs(y.std() - 1) < 0.05
    rep.check(f"{os.path.basename(uri)}: mean 0, SD 1", ok, f"{y.mean():.3f}, {y.std():.3f}")
counts = ["n_table", "n_in_set", "n_implausible", "n_with_pcs", "n_retained"]
rep.table(qc.round(3), "qc_summary", counts=counts)

In [ ]:
fig, axes = plt.subplots(1, len(cfg.lists["phenotypes"]), figsize=(12, 4))
for ax, ph in zip(np.atleast_1d(axes), cfg.lists["phenotypes"]):
    for tr in cfg.lists["transforms"]:
        hist_safe(ax, pd.read_csv(f"{OUT_DIR}/{ph}__{tr}__base_pcs.pheno", sep=" ")["Y"], label=tr)
    ax.set(title=ph, xlabel="residualised, standardised"); ax.legend()
rep.fig(fig, "cleaned_distributions"); plt.show()
rep.save()